In [31]:
import numpy as np
from case_studies import *

# https://en.wikipedia.org/wiki/Wolfe_conditions For c_1 and c_2 choice
#def wolfe_line_search(f, df, x, p, c1=1e-4, c2=0.9, max_iter=1000):
#    a = 1.0
#    fx = f(x)
#    grad_x = df(x)
#    for _ in range(max_iter):
#        new_x = x + a * p
#        if f(new_x) > fx + c1 * a * np.dot(p, grad_x):
#            a *= 0.5  # Reduce step size
#        elif np.dot(p, df(new_x)) < c2 * np.dot(p, grad_x):
#            a *= 2.0  # Increase step size
#        else:
#            break
#    return a

def is_pos_def(H):
    return np.all(np.linalg.eigvals(H) > 0)

def backtrack(f, df, x, p, alpha_init=1.0, rho=0.5, c1=1e-4):
    fx = f(x)
    grad_x = df(x)
    alpha = alpha_init
    while f(x + alpha * p) > fx + c1 * alpha * np.dot(grad_x, p):
        alpha *= rho
    return alpha

def steepest_descent(f, df, x0, c1=1e-6, rho=0.5, max_iter=1000):
    x = x0.copy()
    beta = 1.0
    xs=[]
    xs.append(x)
    for _ in range(max_iter):
        p = -df(x) # p_k = −∇f(x_k);
        alpha = backtrack(f, df, x, p, beta, rho=rho, c1=c1) # α_k = backtrack(x_k,p_k,β_k,c_1,ρ);
        x = x + alpha * p # x_{k+1} = x_k + α_k p_k;
        beta = alpha / rho # β_{k+1} = α_k / ρ;
        xs.append(x)
    return xs

def newtons_method(f, df, Hf, x0, c1=1e-6, rho=0.5, max_iter=1000):
    x = x0.copy()
    xs=[]
    xs.append(x)
    for _ in range(max_iter):
        H = Hf(x)
        if is_pos_def(Hf(x)): # if ∇^2 f(x_k) is positive definite then
            p = np.dot(-np.linalg.inv(Hf(x)), df(x)) # p_k = (−∇^2 f(x_k))^{−1} ∇f(x_k)
        else:
            e_values, e_vectors = np.linalg.eigh(Hf(x)) # Compute eigenvalues λ_i and eigenvector v_i of ∇^2 f(xk);
            new_H = np.zeros_like(H)
            for i in range(len(e_values)): # sum_{i=1}^N
                new_H += (1 / np.abs(e_values[i])) * np.dot(e_vectors[:, i], e_vectors[:, i]) # H=1/|λ_i| * v_i.v_i^T
            p = -new_H @ df(x) # p_k = −H^∇ f(x_k)
        a = backtrack(f, df, x, p, 1.0, rho, c1) # α_k = backtrack(x_k,p_k,1.0,c1,ρ);
        x = x + a * p # x_{k+1} = x_k + α_k p_k;
        xs.append(x)
    return xs

# Steepest Descent function (updated version)
def steepest_descent2(f, df, x0, max_iter=10000, epsilon=1e-6):
    x = x0.copy()
    xs = [x]
    for _ in range(max_iter):
        if np.linalg.norm(df(x)) < epsilon: break
        p = -df(x)
        alpha = backtrack(f, df, x, p, 1.0)
        x = x + alpha * p
        xs.append(x)
    return np.array(xs)


# Newton's Method function (updated version)
def conjugate_gradient(Q, g, epsilon=1e-6):
    x_k = 0
    nabla_f_k = g
    p_k = -nabla_f_k
    for _ in range(len(g)):
        Qp_k = Q @ p_k
        a_k = -(np.transpose(p_k) @ nabla_f_k / (np.transpose(p_k) @ Qp_k))
        x_k = x_k + a_k * p_k
        nabla_f_k = Q @ x_k + g
        
        if np.linalg.norm(nabla_f_k) < epsilon:
            break

        upper = np.transpose(nabla_f_k.T) @ Qp_k
        lower = np.transpose(p_k @ Qp_k)
        
        p_k = -nabla_f_k + (upper/lower) * p_k
    
    return x_k

def newton_cg(f, df, Hf, x_0, c1=1e-4, rho=0.5, max_iter=1000, epsilon=1e-6):
    x_k = x_0
    xs = [x_k]
    
    for _ in range(max_iter):
        nabla_f_k = df(x_k)
        if np.linalg.norm(nabla_f_k) < epsilon:
            break
        
        eta_k = 0.5 * min(0.5, np.sqrt(np.linalg.norm(nabla_f_k)))
        epsilon_k = eta_k * np.linalg.norm(nabla_f_k)
        
        Q_k = Hf(x_k)        
        p_k = conjugate_gradient(Q_k, nabla_f_k, epsilon_k)
        a_k = backtrack(f, df, x_k, p_k, 1.0, rho, c1)
        x_k = x_k + a_k * p_k
        xs.append(x_k)
    
    return np.array(xs)

x0_f1 = np.random.randn(5)
x0_f2 = np.random.randn(2)
x0_f3 = np.random.randn(5)
x0_f4 = np.random.randn(5)
x0_f5 = np.random.randn(5)

print("Steepest Descent Results:")
print("f1:", steepest_descent(f1, df1, x0_f1)[0])
print("f2:", steepest_descent(f2, df2, x0_f2)[0])
print("f3:", steepest_descent(f3, df3, x0_f3)[0])
print("f4:", steepest_descent(f4, df4, x0_f4)[0])
print("f5:", steepest_descent(f5, df5, x0_f5)[0])

# Running Newton's Method
print("\nNewton's Method Results:")
print("f1:", newton_cg(f1, df1, Hf1, x0_f1)[0])
print("f2:", newton_cg(f2, df2, Hf2, x0_f2)[0])
print("f3:", newton_cg(f3, df3, Hf3, x0_f3)[0])
print("f4:", newton_cg(f4, df4, Hf4, x0_f4)[0])
print("f5:", newton_cg(f5, df5, Hf5, x0_f5)[0])

Steepest Descent Results:
f1: [-0.17427201  1.57725948  0.31655968  0.52452661 -0.63476255]
f2: [ 0.44312866 -0.32896994]
f3: [-0.57941328  0.17585307  0.44829952 -1.95753017 -1.88203921]
f4: [ 0.29026923 -0.25940029 -0.9921396  -0.59278676  0.59422831]
f5: [ 0.6487192  -0.26170728 -0.68828394 -0.17846765 -0.06292067]

Newton's Method Results:
f1: [-0.17427201  1.57725948  0.31655968  0.52452661 -0.63476255]
f2: [ 0.44312866 -0.32896994]
f3: [-0.57941328  0.17585307  0.44829952 -1.95753017 -1.88203921]
f4: [ 0.29026923 -0.25940029 -0.9921396  -0.59278676  0.59422831]
f5: [ 0.6487192  -0.26170728 -0.68828394 -0.17846765 -0.06292067]
